# 001 · Running Python Code

> **Syntax › Basics** · Python 3.14

Before writing any Python, it helps to know *how* Python code actually runs. There is no project file, no `Main` method and no separate build step — and that changes how you structure even the smallest program.

**In this notebook you will learn:**

- how CPython compiles to bytecode and interprets it (vs. the C# compiler → IL → JIT pipeline)
- the different ways to run Python: the REPL, scripts, `uv run`, and notebooks
- why Python has no `Main` method, and what the `if __name__ == "__main__":` idiom is for
- how to check which Python version you are running

## C# mental model

| C# / .NET | Python (CPython) |
|---|---|
| `csc` / Roslyn compiles `.cs` → IL in a `.dll` | CPython compiles `.py` → bytecode (cached as `.pyc` in `__pycache__/`) |
| CLR JIT-compiles IL to machine code | the CPython interpreter loop executes bytecode (3.13+ can optionally be built with an *experimental* JIT) |
| explicit build step (`dotnet build`) | no separate build step — compilation happens automatically at import/run time |
| `dotnet run` | `python script.py` or `uv run script.py` |
| `static void Main(string[] args)` / top-level statements | every module runs top to bottom; `if __name__ == "__main__":` guards the entry point |
| `csi` / `dotnet fsi` / C# Interactive | the Python REPL (`python` with no arguments) |
| .NET Interactive / Polyglot notebooks | Jupyter notebooks (what you are reading now) |

The key mental shift: in Python, **running a file means executing its statements one by one, top to bottom**. Defining a function or class is itself a statement that runs.

## Interpreted… but compiled to bytecode

People call Python "interpreted", but CPython (the reference implementation) does compile your source first — to **bytecode**, a stack-based instruction set conceptually similar to IL. The difference is *when*: compilation is implicit and happens just before execution, and the bytecode is then interpreted rather than JIT-compiled to native code.

```csharp
// C# — you can inspect IL with tools like ildasm or ILSpy
static int Add(int a, int b) => a + b;
// IL: ldarg.0, ldarg.1, add, ret
```

Python ships with a disassembler in the standard library, `dis`:

In [ ]:
import dis

def add(a, b):
    return a + b

dis.dis(add)   # the exact opcodes vary between Python versions

The compiled bytecode is attached to the function as a *code object*. You rarely need it, but it shows that a compile step really does exist:

In [ ]:
code = add.__code__
print(type(code).__name__)    # code
print(code.co_varnames)       # ('a', 'b') — local variable names
print(len(code.co_code), "bytes of bytecode")

⚠️ **Gotcha:** because there is no ahead-of-time compile of the whole program, many errors that the C# compiler would catch (a misspelled variable, calling a method that doesn't exist, passing the wrong type) only show up **when that line runs**. Only *syntax* errors are reported before execution. Type checkers such as mypy or pyright fill this gap *(see Type Hints › 001 · Type Annotations)*.

In [ ]:
def broken():
    return undefined_name + 1   # compiles fine — nothing checks names at compile time

print("defined broken() without complaint")

try:
    broken()
except NameError as e:
    print(f"{type(e).__name__}: {e}")

## Ways to run Python

### 1. The REPL

Running `python` (or `uv run python`) with no arguments starts an interactive Read–Eval–Print Loop. Expressions are evaluated and their value is echoed:

```text
$ uv run python
Python 3.14.x ...
>>> 2 + 3
5
>>> name = "alice"
>>> name.upper()
'ALICE'
>>> exit()
```

### 2. Scripts

A script is just a `.py` file. There is no `.csproj` and nothing to compile first.

```bash
# C#
dotnet new console -o Hello
dotnet run --project Hello

# Python — plain interpreter
python hello.py

# Python — via uv, which runs it inside the project's virtual environment
uv run hello.py
uv run python -m some_module    # run a module by name (like `python -m`)
python -c "print('one-liner')"  # run a snippet passed on the command line
```

`uv run` makes sure the right Python version and the project's dependencies are used — think of it as `dotnet run` plus automatic restore *(see Modules & Packages › 003 · Environments and Dependencies with uv)*.

### 3. Notebooks

A Jupyter notebook is a sequence of **cells**. Each code cell is sent to a long-running Python process (the *kernel*), so variables defined in one cell are visible in every cell you run afterwards — in the order you *run* them, not the order they appear on the page.

In [ ]:
greeting = "Hello from an earlier cell"

In [ ]:
print(greeting)   # works because the previous cell has already run in the same kernel

⚠️ **Gotcha:** if you run cells out of order, or delete a cell after running it, the kernel still remembers its variables. When a notebook behaves strangely, use **Restart Kernel & Run All** to make sure it works from top to bottom.

In a notebook, the value of the **last expression** in a cell is displayed automatically (like the REPL echo). Use `print()` when you want output from anywhere else in the cell.

In [ ]:
x = 21
x * 2        # last expression — its value is displayed without print()

## No `Main` method: top-level code

In C# the runtime looks for an entry point — `static void Main` or, since C# 9, top-level statements in exactly one file.

```csharp
// C# — Program.cs
namespace Demo;

class Program
{
    static void Main(string[] args)
    {
        Console.WriteLine("Hello!");
    }
}
```

Python has no entry-point method. When you run a file, the interpreter simply executes every statement at the top level of that file. `def` and `class` are statements too — they create a function/class object and bind it to a name at the moment they run.

In [ ]:
print("1. top-level statement runs immediately")

def say_hello():
    print("3. function body runs only when called")

print("2. the def above just created a function object:", say_hello)
say_hello()

## The `if __name__ == "__main__":` idiom

Every module has a built-in variable `__name__`:

- when a file is **run directly**, `__name__` is set to `"__main__"`
- when a file is **imported** by another module, `__name__` is the module's name (e.g. `"greetings"`)

Because importing a module *runs* its top-level code, you guard "script-only" code with `if __name__ == "__main__":`. This lets a single file be both an importable library and a runnable program — roughly the role `Main` plays in C#.

In [ ]:
print(__name__)   # in a notebook, the kernel's code runs as "__main__" too

Let's prove it by writing two small files into a temporary directory and running them with a separate Python process. We use `sys.executable` (the path of the current interpreter) with `subprocess` — the equivalent of `Process.Start` in C#.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

greetings_src = textwrap.dedent('''
    def greet(name):
        return f"Hello, {name}!"

    print(f"[greetings] top-level code runs, __name__ = {__name__!r}")

    if __name__ == "__main__":
        print("[greetings] running as a script:", greet("alice"))
''')

app_src = textwrap.dedent('''
    import greetings          # importing runs greetings' top-level code

    print("[app] imported:", greetings.greet("bob"))
''')

with tempfile.TemporaryDirectory() as tmp:
    folder = Path(tmp)
    (folder / "greetings.py").write_text(greetings_src)
    (folder / "app.py").write_text(app_src)

    for script in ["greetings.py", "app.py"]:
        print(f"$ python {script}")
        result = subprocess.run([sys.executable, script], cwd=folder,
                                capture_output=True, text=True)
        print(result.stdout)

Notice the difference:

- running `greetings.py` directly sets `__name__` to `"__main__"`, so the guarded block runs
- when `app.py` imports it, `__name__` is `"greetings"`, so the top-level `print` still runs but the guarded block does **not**

A common convention is to put the script logic in a `main()` function and call it from the guard:

```python
def main() -> None:
    ...

if __name__ == "__main__":
    main()
```

⚠️ **Gotcha:** any unguarded top-level code (printing, opening files, starting servers) runs as a side effect every time someone imports your module. Keep top-level code to definitions and constants.

## The Zen of Python

Python's design philosophy is captured in PEP 20, "The Zen of Python". It ships as an Easter egg: importing the `this` module prints it. Note that the text appears because importing a module executes its top-level code — exactly what we just discussed!

In [ ]:
import this

A few lines worth remembering as a C# developer: *"Explicit is better than implicit"*, *"Readability counts"*, and *"There should be one-- and preferably only one --obvious way to do it."* You will see these ideas behind many of the conventions in later notebooks.

Run `import this` a second time and nothing is printed: a module's top-level code runs only on the **first** import in a process, after which it is cached in `sys.modules` *(see Modules & Packages › 001 · Importing Modules)*.

## Which Python am I running?

The equivalent of `Environment.Version` / `RuntimeInformation.FrameworkDescription`:

```csharp
// C#
Console.WriteLine(Environment.Version);
Console.WriteLine(System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription);
```

In [ ]:
import platform
import sys

print(sys.version)                    # full version string
print(sys.version_info)               # a named tuple — easy to compare
print(sys.version_info >= (3, 14))    # feature checks: compare against a tuple
print(platform.python_implementation())   # CPython (others: PyPy, ...)
print(sys.executable)                 # path of the interpreter running this code

From a terminal you can use `python --version` or `uv run python --version` (like `dotnet --version`).

## Summary

| Task | C# | Python |
|---|---|---|
| Compile | `dotnet build` (→ IL) | automatic, implicit (→ bytecode `.pyc`) |
| Inspect low-level code | ILSpy / ildasm | `dis.dis(func)` |
| Run a program | `dotnet run` | `python app.py` / `uv run app.py` |
| Interactive shell | C# Interactive / `dotnet fsi` | `python` (REPL) |
| Entry point | `static void Main` / top-level statements | module top-level code + `if __name__ == "__main__":` |
| Start another process | `Process.Start(...)` | `subprocess.run([...])` |
| Runtime version | `Environment.Version` | `sys.version` / `sys.version_info` |

## Exercises

1. Write a function `describe()` that returns the string `"Python X.Y"` using `sys.version_info` (e.g. `"Python 3.14"`).
2. Using a `TemporaryDirectory`, write a script `calc.py` that defines `square(n)` and, **only when run directly**, prints `square(7)`. Run it with `subprocess` and print its output.
3. Without running it, predict what this prints, then check: `print(__name__ == "__main__")` inside a notebook cell.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
import sys

def describe() -> str:
    major, minor = sys.version_info[:2]
    return f"Python {major}.{minor}"

print(describe())

# 2
import subprocess, sys, tempfile
from pathlib import Path

src = '''
def square(n):
    return n * n

if __name__ == "__main__":
    print(square(7))
'''

with tempfile.TemporaryDirectory() as tmp:
    script = Path(tmp) / "calc.py"
    script.write_text(src)
    out = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
    print(out.stdout)   # 49

# 3 — True: the notebook kernel executes cells in the "__main__" module.
```

</details>